In [8]:
from flask import Flask, jsonify, request
from flask_cors import CORS
import pymysql  # or import mysql.connector
import joblib
import pandas as pd
import numpy as np
# calories, protien, sugar, fat, fiber, carbohydrates

In [9]:
model = joblib.load("SleepAnalysis2.pkl")
# /home/kali/College/Mini/Job/SleepAnalysis.pkl
# SleepAnalysis2.pkl
db = pymysql.connect(
host = "localhost",
user = "root", #root #aditya
password = "root",
database = "mini"
)
cursor = db.cursor()

In [ ]:
app = Flask(__name__)
cors = CORS(app, origins = '*')
@app.route("/submit", methods = ['GET', 'POST'] )
def submit():
    data = request.get_json()
    age = int(data['age'])
    bed_time = data['bedTime']
    wake_time = data['wakeTime']
    awakenings = float(data['awakenings'])
    caffeine = float(data['caffeine'])
    alcohol = float(data['alcohol'])
    smoking = "Yes" if data['smoking'].lower() == "yes" else "No"  # Store as Yes/No
    exercise = float(data['exercise'])
    REM = int(data['REM'])
    deep_sleep = int(data['deep_sleep'])


    smoking_numeric = 1 if smoking == "Yes" else 0
    sleep_duration = (float(wake_time.split(":")[0]) - float(bed_time.split(":")[0]) + 24) % 24

    userDataDF = np.array([[  age,
 sleep_duration,
    REM, 
  deep_sleep, 
awakenings,
 caffeine,
 alcohol,
 smoking_numeric,  
exercise]])
    # userDataDF = pd.DataFrame({
    #     'Age': [age],
    #     'Sleep_duration': [sleep_duration],
    #     'REM_sleep_percentage': [REM], 
    #     'Deep_sleep_percentage': [deep_sleep], 
    #     'Awakenings': [awakenings],
    #     'Caffeine_consumption': [caffeine],
    #     'Alcohol_consumption': [alcohol],
    #     'Smoking_status': [smoking_numeric],  
    #     'Exercise_frequency': [exercise]
    # })
    prediction = model.predict(userDataDF)
    prediction[0] = np.expm1(prediction[0])
    sleep_efficiency = prediction[0]
    insert_query = """
        INSERT INTO sleep_data (age, bed_time, wake_time, awakenings, caffeine, alcohol, smoking, exercise, sleep_efficiency, REM_percentage, deep_sleep_percentage) 
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    values = (age, bed_time, wake_time, awakenings, caffeine, alcohol, smoking, exercise, sleep_efficiency, REM, deep_sleep)
    cursor.execute(insert_query, values)
    db.commit()
    print(data)
    return jsonify({'sleep_efficiency': prediction[0], "duration": sleep_duration, "data" : data, "values": values })

@app.route("/diet", methods = ['GET', 'POST'])
def diet():
    data = request.get_json()
    food_query = f"{data['food']}%"
    cursor.execute("select name from dietdb where name like %s limit 10", (food_query,))
    results = cursor.fetchall()
    results = [i[0] for i in results]
    return jsonify({'results' : results})

@app.route("/diet/output", methods = ['GET', 'POST'])
def output():
    data = request.get_json()
    serving = int(data['serving']) / 100
    cursor.execute("select calories, protein, carbohydrate, cholesterol, total_fat, sugars from dietdb where name = %s", (data['food'],))
    result = cursor.fetchone()
    output = {
        "calories": int(result[0] * serving),
        "protein": int(result[1] * serving),
        "carbohydrate": int(result[2] * serving),
        "cholesterol": int(result[3] * serving),
        "total_fat": int(result[4] * serving),
        "sugars": int(result[5] * serving)
    }
    print(type(output))
    print(output)
    return jsonify(output)

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [30/Mar/2025 13:50:51] "OPTIONS /submit HTTP/1.1" 200 -
c:\Users\HP\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
127.0.0.1 - - [30/Mar/2025 13:50:52] "POST /submit HTTP/1.1" 200 -


{'age': '22', 'bedTime': '13:51', 'wakeTime': '06:49', 'awakenings': '1', 'caffeine': '2', 'alcohol': '1', 'smoking': 'No', 'exercise': '2', 'REM': '15', 'deep_sleep': '50'}


127.0.0.1 - - [30/Mar/2025 13:52:51] "OPTIONS /submit HTTP/1.1" 200 -
c:\Users\HP\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
127.0.0.1 - - [30/Mar/2025 13:52:51] "POST /submit HTTP/1.1" 200 -


{'age': '22', 'bedTime': '13:51', 'wakeTime': '06:49', 'awakenings': '1', 'caffeine': '2', 'alcohol': '1', 'smoking': 'No', 'exercise': '2', 'REM': '15', 'deep_sleep': '50'}


127.0.0.1 - - [30/Mar/2025 13:55:59] "OPTIONS /submit HTTP/1.1" 200 -
c:\Users\HP\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
127.0.0.1 - - [30/Mar/2025 13:56:00] "POST /submit HTTP/1.1" 200 -


{'age': '44', 'bedTime': '01:55', 'wakeTime': '07:55', 'awakenings': '0', 'caffeine': '0', 'alcohol': '0', 'smoking': 'No', 'exercise': '2', 'REM': '18', 'deep_sleep': '80'}


127.0.0.1 - - [30/Mar/2025 13:58:55] "OPTIONS /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 13:58:55] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:27] "OPTIONS /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:27] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:30] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:32] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:34] "OPTIONS /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:34] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:36] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:39] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:41] "OPTIONS /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:41] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:42] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:44] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [30/Mar/2025 14:02:44] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - -